# Module 2: Policy Competition on OpenSpiel

Build 4 policies, compete them on Catch, then switch to another game with the same code.

**Time:** ~20 min · **Difficulty:** Beginner · **GPU:** Not required

In [2]:
# Install dependencies
!uv pip install -q "git+https://github.com/meta-pytorch/OpenEnv.git@v0.2.2#subdirectory=envs/openspiel_env"

## 1. Connect to Catch

Catch: a ball falls from the top of a 10×5 grid. Move your paddle to catch it.

- Actions: `0` = LEFT, `1` = STAY, `2` = RIGHT
- Reward: `+1` if caught, `0` if missed

In [1]:
from openspiel_env import OpenSpielEnv
from openspiel_env.models import OpenSpielAction, OpenSpielObservation
import random

CATCH_URL = "https://mroyme-openspiel-env.hf.space"

# Quick sanity check
with OpenSpielEnv(base_url=CATCH_URL).sync() as env:
    result = env.reset()
    print(f"Legal actions: {result.observation.legal_actions}")
    print(f"Info state shape: {len(result.observation.info_state)} values")
    print(f"Game phase: {result.observation.game_phase}")
    print(vars(result.observation))

Legal actions: [0, 1, 2]
Info state shape: 50 values
Game phase: initial
{'done': False, 'reward': None, 'metadata': {}, 'info_state': [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0], 'legal_actions': [0, 1, 2], 'game_phase': 'initial', 'current_player_id': 0, 'opponent_last_action': None}


## 2. Define Four Policies

Each policy takes an `OpenSpielObservation` and returns an action ID.

In [ ]:
class RandomPolicy:
    """Pure random — baseline."""
    name = "Random"

    def select_action(self, obs: OpenSpielObservation) -> int:
        return random.choice(obs.legal_actions)


class AlwaysStayPolicy:
    """Never moves — hopes ball lands on paddle."""
    name = "Always Stay"

    def select_action(self, obs: OpenSpielObservation) -> int:
        return 1  # STAY


class SmartPolicy:
    """Moves paddle toward ball — optimal for Catch."""
    name = "Smart Heuristic"

    def select_action(self, obs: OpenSpielObservation) -> int:
        info_state = obs.info_state
        grid_width = 5

        # Find ball column (first 1.0 in the flattened grid)
        ball_col = None
        for idx, val in enumerate(info_state):
            if abs(val - 1.0) < 0.01:
                ball_col = idx % grid_width
                break

        # Paddle is in the last row
        last_row = info_state[-grid_width:]
        paddle_col = next((i for i, v in enumerate(last_row) if abs(v - 1.0) < 0.01), None)

        if ball_col is not None and paddle_col is not None:
            if paddle_col < ball_col:
                return 2  # RIGHT
            elif paddle_col > ball_col:
                return 0  # LEFT
        return 1  # STAY


class LearningPolicy:
    """Epsilon-greedy — starts random, learns to be smart."""
    name = "Epsilon-Greedy"

    def __init__(self):
        self.steps = 0
        self._smart = SmartPolicy()

    def select_action(self, obs: OpenSpielObservation) -> int:
        self.steps += 1
        #10% of the time, a random action is chosen; 90% of the time, SmartPolicy is used.
        epsilon = max(0.1, 1.0 - self.steps / 100)
        if random.random() < epsilon:
            return random.choice(obs.legal_actions)
        return self._smart.select_action(obs)


print("Policies defined: Random, Always Stay, Smart Heuristic, Epsilon-Greedy")

Policies defined: Random, Always Stay, Smart Heuristic, Epsilon-Greedy


## 3. Run a Single Episode

Helper to play one full game and return whether the ball was caught.

In [4]:
def run_episode(env, policy, verbose=False):
    """Play one episode. Returns 1 if caught, 0 if missed."""
    result = env.reset()
    step = 0

    while not result.done:
        action_id = policy.select_action(result.observation)
        if verbose:
            name = {0: "LEFT", 1: "STAY", 2: "RIGHT"}.get(action_id, str(action_id))
            print(f"  Step {step}: {name}")
        result = env.step(OpenSpielAction(action_id=action_id, game_name="catch"))
        step += 1

    caught = 1 if result.reward and result.reward > 0 else 0
    if verbose:
        print(f"  Result: {'Caught!' if caught else 'Missed'} (reward={result.reward})")
    return caught


# Demo: one verbose episode with SmartPolicy
with OpenSpielEnv(base_url=CATCH_URL).sync() as env:
    print("Smart policy — single episode:")
    run_episode(env, SmartPolicy(), verbose=True)

Smart policy — single episode:
  Step 0: RIGHT
  Step 1: STAY
  Step 2: STAY
  Step 3: STAY
  Step 4: STAY
  Step 5: STAY
  Step 6: STAY
  Step 7: STAY
  Step 8: STAY
  Result: Caught! (reward=1.0)


## 4. Policy Competition

Run 50 episodes per policy and compare success rates.

In [5]:
NUM_EPISODES = 50
#each policy is run for 50 episodes, Each episode lasts until the ball reaches the bottom row (where the paddle is).
#Since the grid has 10 rows, the maximum number of steps in one episode is 10 (the ball falls one row per step).

policies = [
    RandomPolicy(),
    AlwaysStayPolicy(),
    SmartPolicy(),
    LearningPolicy(),
]

results = {}

with OpenSpielEnv(base_url=CATCH_URL).sync() as env:
    for policy in policies:
        wins = sum(run_episode(env, policy) for _ in range(NUM_EPISODES))
        rate = wins / NUM_EPISODES * 100
        results[policy.name] = rate
        print(f"{policy.name:20s} — {rate:5.1f}% ({wins}/{NUM_EPISODES})")

print("\n--- Results ---")
for name, rate in sorted(results.items(), key=lambda x: -x[1]):
    bar = "█" * int(rate / 2)
    print(f"{name:20s} [{bar:<50}] {rate:.1f}%")

Random               —  12.0% (6/50)
Always Stay          —  16.0% (8/50)
Smart Heuristic      — 100.0% (50/50)
Epsilon-Greedy       —  84.0% (42/50)

--- Results ---
Smart Heuristic      [██████████████████████████████████████████████████] 100.0%
Epsilon-Greedy       [██████████████████████████████████████████        ] 84.0%
Always Stay          [████████                                          ] 16.0%
Random               [██████                                            ] 12.0%


Expected results:
- **Random**: ~20% (pure luck)
- **Always Stay**: ~20% (terrible strategy)
- **Smart Heuristic**: ~100% (optimal)
- **Epsilon-Greedy**: ~80-90% (improves over episodes)

## 5. Switch Games

The same `OpenSpielEnv` client works for all 6 OpenSpiel games. Let's try Tic-Tac-Toe — the observation format is identical, only the game logic changes.

In [6]:
TTT_URL = "https://mroyme-openspiel-env.hf.space"

with OpenSpielEnv(base_url=TTT_URL).sync() as env:
    result = env.reset()
    print(f"Game: Tic-Tac-Toe")
    print(f"Legal actions: {result.observation.legal_actions}")
    print(f"Info state: {result.observation.info_state}")
    print(f"Current player: {result.observation.current_player_id}")
    print()

    # Play randomly until game ends
    step = 0
    while not result.done:
        action_id = random.choice(result.observation.legal_actions)
        result = env.step(OpenSpielAction(action_id=action_id, game_name="tic_tac_toe"))
        step += 1
        print(f"Step {step}: action={action_id}, reward={result.reward}, done={result.done}")

    print(f"\nGame over! Final reward: {result.reward}")

Game: Tic-Tac-Toe
Legal actions: [0, 1, 2]
Info state: [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0]
Current player: 0

Step 1: action=0, reward=0.0, done=False
Step 2: action=2, reward=0.0, done=False
Step 3: action=1, reward=0.0, done=False
Step 4: action=1, reward=0.0, done=False
Step 5: action=0, reward=0.0, done=False
Step 6: action=1, reward=0.0, done=False
Step 7: action=2, reward=0.0, done=False
Step 8: action=2, reward=0.0, done=False
Step 9: action=0, reward=-1.0, done=True

Game over! Final reward: -1.0


Same client class. Same observation type. Different game. That's the OpenEnv promise.

## Summary

- Built 4 policies with increasing sophistication
- Ran a 50-episode competition on Catch
- Switched to Tic-Tac-Toe with zero code changes to the client

All policies work with `OpenSpielObservation` — you read `info_state`, `legal_actions`, and `done`. The game logic is on the server. Your code is on the client.

**Next:** [Module 3](../module-3/README.md) — Deploying environments to HF Spaces.